Audyt 2026-09-10: jawne wejście, brak imputacji, zapisy tylko w audit_v3. Ten notebook dokumentuje historyczną klasyfikację tagów; eksperyment 03 używa istniejącej wersji tag_classification_v1.xlsx. Szczegóły: ../docs/model_audit.md.


In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, r2_score

pd.set_option("display.max_columns", None)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

print(PROJECT_ROOT)
print(DATA_PROCESSED)
import sys
sys.path.insert(0, str(PROJECT_ROOT))
from src.model_features import clean_inputs
AUDIT_OUTPUT = DATA_PROCESSED / "audit_v3"
AUDIT_OUTPUT.mkdir(exist_ok=True)


In [ ]:
df = pd.read_parquet(DATA_PROCESSED / "dataset_clean.parquet")
print(df.shape)
df.head()


In [ ]:
print(type(df.index))
print(df.index.min())
print(df.index.max())
print(df.index.to_series().diff().value_counts().head())

In [ ]:
df.columns.tolist()

In [ ]:
df.info()

In [ ]:
missing_summary = df.isna().sum().sort_values(ascending=False)
missing_summary[missing_summary > 0]

In [ ]:
missing_percent = (df.isna().sum() / len(df) * 100).sort_values(ascending=False)

missing_table = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": missing_percent
}).sort_values("missing_count", ascending=False)

missing_table[missing_table["missing_count"] > 0]

In [ ]:
cols_with_many_missing = ["016A00396", "016A00219"]

for col in cols_with_many_missing:
    missing_idx = df.index[df[col].isna()]
    
    print("\n", col)
    print("Number of missing:", len(missing_idx))
    print("First missing:", missing_idx.min())
    print("Last missing:", missing_idx.max())

In [ ]:
df[["016A00396", "016A00219"]].isna().value_counts()


In [ ]:
plt.figure(figsize=(14, 4))

plt.plot(df.index, df["016A00396"].isna().astype(int), label="016A00396 missing")
plt.plot(df.index, df["016A00219"].isna().astype(int), label="016A00219 missing", alpha=0.7)

plt.title("Missing data pattern for main columns")
plt.xlabel("Time")
plt.ylabel("Missing flag")
plt.legend()
plt.grid(True)
plt.show()

Tag mapping

In [ ]:
TAG_MAPPING_PATH = DATA_PROCESSED / "tag_mapping.csv"

tag_map = pd.read_csv(TAG_MAPPING_PATH, sep=";")

tag_map.head()

In [ ]:
tag_map.shape

In [ ]:
df_tags = pd.DataFrame({
    "tag": df.columns
})

tag_map_short = tag_map.rename(columns={
    "Nazwa": "tag",
    "Opis": "description",
    "Symbol": "symbol",
    "Jednostka": "unit"
})

df_tags_mapped = df_tags.merge(
    tag_map_short,
    on="tag",
    how="left"
)

df_tags_mapped

In [ ]:
df_tags_mapped[df_tags_mapped["description"].isna()]

In [ ]:
df_cols = set(df.columns)
map_tags = set(tag_map["Nazwa"])

missing_description = sorted(df_cols - map_tags)
not_in_parquet = sorted(map_tags - df_cols)

print("Liczba kolumn w parquet:", len(df_cols))
print("Liczba tagów w mapowaniu:", len(map_tags))

print("\nKolumny z parquet bez opisu w mapowaniu:", len(missing_description))
print(missing_description)

print("\nTagi z mapowania, których nie ma w parquet:", len(not_in_parquet))
print(not_in_parquet)

In [ ]:
# Przygotowanie mapowania tag -> opis / symbol / jednostka

tag_map_short = tag_map.rename(columns={
    "Nazwa": "tag",
    "Opis": "description",
    "Symbol": "symbol",
    "Jednostka": "unit"
})

df_tags_mapped = (
    pd.DataFrame({"tag": df.columns})
    .merge(tag_map_short, on="tag", how="left")
)

df_tags_mapped

In [ ]:
# Wersja bez kolumny technicznej source_file

df_tags_mapped_signals = df_tags_mapped[df_tags_mapped["tag"] != "source_file"].copy()

df_tags_mapped_signals

In [ ]:
duplicates = tag_map[tag_map["Nazwa"].duplicated(keep=False)].sort_values("Nazwa")
duplicates

In [ ]:
print("Liczba wierszy w tag_map:", len(tag_map))
print("Liczba unikalnych tagów:", tag_map["Nazwa"].nunique())
print("Liczba duplikatów:", len(tag_map) - tag_map["Nazwa"].nunique())

In [ ]:
tag_map_unique = tag_map.drop_duplicates(subset="Nazwa", keep="first").copy()

In [ ]:
duplicates = tag_map[tag_map["Nazwa"].duplicated(keep=False)].sort_values("Nazwa")

duplicates

In [ ]:
tag_map["Nazwa"].value_counts()[tag_map["Nazwa"].value_counts() > 1]

In [ ]:
tag_map_unique = tag_map.drop_duplicates(subset="Nazwa", keep="first").copy()

print("Wiersze przed:", len(tag_map))
print("Unikalne tagi po usunięciu duplikatów:", len(tag_map_unique))
print("Liczba unikalnych tagów:", tag_map_unique["Nazwa"].nunique())

In [ ]:
tag_map_short = tag_map_unique.rename(columns={
    "Nazwa": "tag",
    "Opis": "description",
    "Symbol": "symbol",
    "Jednostka": "unit"
})

df_tags_mapped = (
    pd.DataFrame({"tag": df.columns})
    .merge(tag_map_short, on="tag", how="left")
)

df_tags_mapped

In [ ]:
print("Liczba kolumn w df:", len(df.columns))
print("Liczba wierszy po merge:", len(df_tags_mapped))

df_tags_mapped[df_tags_mapped["description"].isna()]

In [ ]:
df_tags_mapped_signals = df_tags_mapped[df_tags_mapped["tag"] != "source_file"].copy()

df_tags_mapped_signals = df_tags_mapped_signals.sort_values("symbol").reset_index(drop=True)

df_tags_mapped_signals

In [ ]:
df_tags_mapped_signals = df_tags_mapped[df_tags_mapped["tag"] != "source_file"].copy()

df_tags_mapped_signals = df_tags_mapped_signals[[
    "tag", "description", "symbol", "unit"
]].copy()

df_tags_mapped_signals

In [ ]:
df_tags_mapped_signals["category"] = ""
df_tags_mapped_signals["role"] = ""
df_tags_mapped_signals["comment"] = ""

df_tags_mapped_signals

In [ ]:
TAG_CLASSIFICATION_PATH = AUDIT_OUTPUT / "tag_classification.xlsx"

df_tags_mapped_signals.to_excel(TAG_CLASSIFICATION_PATH, index=False)

print(TAG_CLASSIFICATION_PATH)

In [ ]:
def classify_tag(row):
    desc = str(row["description"]).lower()
    symbol = str(row["symbol"]).lower()
    unit = str(row["unit"]).lower()

    text = desc + " " + symbol

    # target
    if "pył" in text or "pyl" in text:
        return "target_candidate", "target", "pomiar stężenia pyłu - kandydat na zmienną wyjściową"

    # ESP electrical
    if "zespół zasil wn" in desc or "_wn_" in symbol:
        if "nap" in desc or "uwt" in symbol or "uz" in symbol:
            return "esp_voltage", "input", "napięcie zasilacza WN / napięcie wtórne"
        if "prąd" in desc or "iwt" in symbol or "iz" in symbol:
            return "esp_current", "input", "prąd zasilacza WN / prąd wtórny"
        if "moc" in desc or "nz_wn" in symbol:
            return "esp_power", "input", "moc zespołu zasilania WN"
        if "częst przeskoków" in desc or "fp_wn" in symbol:
            return "esp_spark_rate", "input", "częstość przeskoków"
        if "potw zał" in desc or "zal" in symbol:
            return "esp_status", "input", "status załączenia zasilacza WN"

    # rapping / strzepywanie
    if "strzep" in text:
        return "esp_rapping", "input", "praca układu strzepywania elektrod"

    # hopper / heating / insulation
    if "ogrz" in text or "izol" in text:
        return "esp_heating_status", "input", "status ogrzewania lejów/izolatorów"

    if "leju zsypowym" in desc or "rurociągu" in desc:
        return "esp_temperature", "input", "temperatura leja zsypowego lub rurociągu"

    # flue gas / combustion
    if "spalin" in desc or "spa" in symbol:
        if "o2" in text:
            return "flue_gas_o2", "input", "zawartość tlenu w spalinach"
        if "so2" in text:
            return "flue_gas_so2", "input", "stężenie SO2 w spalinach"
        if "nox" in text:
            return "flue_gas_nox", "input", "stężenie NOx w spalinach"
        if "co " in text or "_co_" in symbol:
            return "flue_gas_co", "input", "stężenie CO w spalinach"
        if "wilgotność" in desc:
            return "flue_gas_humidity", "input", "wilgotność spalin"
        if "f spalin" in desc:
            return "flue_gas_flow", "input", "przepływ spalin"
        if "t spalin" in desc:
            return "flue_gas_temperature", "input", "temperatura spalin"
        if "p spalin" in desc:
            return "flue_gas_pressure", "input", "ciśnienie spalin"

        return "flue_gas_process", "input", "zmienna procesowa związana ze spalinami"

    # boiler / generator / air
    if "generatora" in desc or "moc czynna" in desc:
        return "boiler_load", "input", "obciążenie bloku/kotła/generatora"

    if "pow.do kotła" in desc:
        return "boiler_air_flow", "input", "przepływ powietrza do kotła"

    if "wody zasilającej" in desc:
        return "boiler_water_temperature", "input", "temperatura wody zasilającej"

    if "kierownic" in desc:
        return "boiler_air_damper", "input", "położenie kierownic / element regulacyjny powietrza"

    if "pal. olej" in desc:
        return "burner_status", "input", "status palnika olejowego"

    return "other", "review", "do ręcznego sprawdzenia"


classified = df_tags_mapped_signals.copy()

classified[["category", "role", "comment"]] = classified.apply(
    classify_tag,
    axis=1,
    result_type="expand"
)

classified

In [ ]:
classified[classified["category"] == "other"]

In [ ]:
classified.loc[classified["tag"] == "008A01350", ["category", "role", "comment"]] = [
    "flue_gas_o2",
    "input",
    "stężenie O2 w spalinach / zmienna procesowa spalania"
]

classified.loc[classified["tag"] == "008A01341", ["category", "role", "comment"]] = [
    "flue_gas_so2",
    "input",
    "stężenie SO2 w spalinach / zmienna procesowa emisji"
]

classified.loc[classified["tag"] == "008A01343", ["category", "role", "comment"]] = [
    "flue_gas_nox",
    "input",
    "stężenie NOx w spalinach / zmienna procesowa emisji"
]

classified.loc[classified["tag"] == "008A01347", ["category", "role", "comment"]] = [
    "flue_gas_co",
    "input",
    "stężenie CO w spalinach / zmienna procesowa emisji"
]

classified[classified["category"] == "other"]

In [ ]:
target_candidates = classified[
    classified["description"].str.contains(
        "pył|pyl|zapylen|zapylenie|emisj|dust",
        case=False,
        na=False
    ) |
    classified["symbol"].str.contains(
        "pyl|pył|dust|emis",
        case=False,
        na=False
    )
]

target_candidates

In [ ]:
target_col = "008A01345"

classified.loc[classified["tag"] == target_col, ["category", "role", "comment"]] = [
    "target",
    "target",
    "stężenie pyłu BC1 - zmienna wyjściowa modelu"
]

classified[classified["role"] == "target"]

In [ ]:
input_cols = classified.loc[
    classified["role"] == "input",
    "tag"
].tolist()

print("Target:", target_col)
print("Liczba wejść:", len(input_cols))
print(input_cols)

In [ ]:
target_col in input_cols

In [ ]:
classified[classified["role"] == "input"]["category"].value_counts()

In [ ]:
classified[classified["role"] == "input"][
    ["tag", "description", "symbol", "unit", "category"]
].sort_values("category")

In [ ]:
selected_categories_v1 = [
    "esp_voltage",
    "esp_current",
    "esp_power",
    "esp_spark_rate",
    "esp_status",
    "esp_rapping",
    "esp_temperature",
    "esp_heating_status",
    "burner_status",

    "boiler_load",
    "boiler_air_flow",
    "boiler_air_damper",
    "boiler_water_temperature",

    "flue_gas_temperature",
    "flue_gas_pressure",
    "flue_gas_flow",
    "flue_gas_o2",
    "flue_gas_humidity",
    "flue_gas_so2",
    "flue_gas_co",
    "flue_gas_nox",
]

input_cols_v1 = classified.loc[
    (classified["role"] == "input") &
    (classified["category"].isin(selected_categories_v1)),
    "tag"
].tolist()

print("Target:", target_col)
print("Liczba wejść V1:", len(input_cols_v1))
print(input_cols_v1)

In [ ]:
set(classified.loc[classified["role"] == "input", "tag"]) - set(input_cols_v1)

In [ ]:
df_model = df[[target_col] + input_cols_v1].copy()

print("df_model shape:", df_model.shape)
print("Target:", target_col)
print("Number of input features:", len(input_cols_v1))

missing_model = pd.DataFrame({
    "missing_count": df_model.isna().sum(),
    "missing_percent": df_model.isna().mean() * 100
}).sort_values("missing_count", ascending=False)

missing_model[missing_model["missing_count"] > 0]

In [ ]:
classified[classified["tag"].isin(["016A00219", "016A00396"])][
    ["tag", "description", "symbol", "unit", "category", "role", "comment"]
]

In [ ]:
df_model_clean = df_model.copy()

print("Before cleaning:", df_model_clean.shape)

# Usuwamy rekordy, gdzie brakuje ważnych zmiennych obciążenia
df_model_clean = df_model_clean.dropna(subset=["016A00219", "016A00396"])

print("After dropping missing boiler load rows:", df_model_clean.shape)
print("Removed rows:", len(df_model) - len(df_model_clean))
print("Removed percent:", round((len(df_model) - len(df_model_clean)) / len(df_model) * 100, 3), "%")

In [ ]:
missing_after_main_drop = pd.DataFrame({
    "missing_count": df_model_clean.isna().sum(),
    "missing_percent": df_model_clean.isna().mean() * 100
}).sort_values("missing_count", ascending=False)

missing_after_main_drop[missing_after_main_drop["missing_count"] > 0]

In [ ]:
small_missing_cols = df_model_clean.columns[
    (df_model_clean.isna().sum() > 0) &
    (df_model_clean.isna().sum() <= 5)
].tolist()

classified[classified["tag"].isin(small_missing_cols)][
    ["tag", "description", "symbol", "unit", "category", "role", "comment"]
].sort_values("tag")

In [ ]:
# No forward/backward filling of labels or industrial status signals.
before_complete_case = len(df_model_clean)
df_model_clean = clean_inputs(df_model, input_cols_v1)
print("Rows removed after boiler filtering:", before_complete_case - len(df_model_clean))
print("Remaining missing cells:", df_model_clean.isna().sum().sum())


In [ ]:
for col in small_missing_cols:
    print("\n", col)
    print(classified.loc[classified["tag"] == col, ["description", "category"]].to_string(index=False))
    print("Unique values:", sorted(df_model_clean[col].dropna().unique()))

In [ ]:
MODEL_CLEAN_PATH = AUDIT_OUTPUT / "df_model_clean_v1.parquet"

df_model_clean.to_parquet(MODEL_CLEAN_PATH)

print(MODEL_CLEAN_PATH)
print(df_model_clean.shape)

In [ ]:
TAG_CLASSIFICATION_FINAL_PATH = AUDIT_OUTPUT / "tag_classification_v1.xlsx"

classified.to_excel(TAG_CLASSIFICATION_FINAL_PATH, index=False)

print(TAG_CLASSIFICATION_FINAL_PATH)

Rzeczy zrobione:

mapowanie tagów
klasyfikacja tagów
wybór targetu
wybór wejść
czyszczenie braków
zapis df_model_clean_v1.parquet